In [0]:
%run "/Workspace/DAV Project/YouTube-Trending-Analytics/src/pipelines/silver_to_gold"

In [0]:
#Check the source
from pyspark.sql import functions as F

SILVER = "workspace.youtube_trending.silver_video_daily"

silver = spark.table(SILVER)

print("Silver rows:", silver.count())

silver.select(
    F.min("snapshot_date").alias("earliest_date"),
    F.max("snapshot_date").alias("latest_date"),
    F.countDistinct("snapshot_date").alias("snapshot_dates")
).show()

Silver rows: 7059
+-------------+-----------+--------------+
|earliest_date|latest_date|snapshot_dates|
+-------------+-----------+--------------+
|   2026-09-26| 2026-10-25|            14|
+-------------+-----------+--------------+



In [0]:
import uuid

gold_result = run_silver_to_gold(
    spark=spark,
    start_date="2026-09-26",
    end_date="2026-10-25",
    load_type="full",
    batch_id=f"gold_{uuid.uuid4().hex[:8]}"
)

print(gold_result)

{'batch_id': 'gold_7a1cb722', 'source_rows': 7059, 'fact_rows_for_range': 7059, 'gold_fact_table': 'workspace.youtube_trending.gold_fact_video_daily', 'gold_dimensions': ['workspace.youtube_trending.gold_dim_date', 'workspace.youtube_trending.gold_dim_region', 'workspace.youtube_trending.gold_dim_category', 'workspace.youtube_trending.gold_dim_channel', 'workspace.youtube_trending.gold_dim_video']}


In [0]:
#Check for duplicate fact records
from pyspark.sql import functions as F

fact = spark.table("workspace.youtube_trending.gold_fact_video_daily")

duplicates = (
    fact.groupBy("video_id", "region_code", "snapshot_date")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate fact keys:", duplicates.count())

Duplicate fact keys: 0


In [0]:
from pyspark.sql import functions as F

# 1. Check row counts
tables = [
    "gold_dim_date",
    "gold_dim_region",
    "gold_dim_category",
    "gold_dim_channel",
    "gold_dim_video",
    "gold_fact_video_daily"
]

for name in tables:
    df = spark.table(f"workspace.youtube_trending.{name}")
    print(f"{name}: {df.count():,} rows")

# 2. Check for null fact keys
fact = spark.table(
    "workspace.youtube_trending.gold_fact_video_daily"
)

fact.select(
    F.sum(F.col("video_id").isNull().cast("int")).alias("null_video_id"),
    F.sum(F.col("region_code").isNull().cast("int")).alias("null_region"),
    F.sum(F.col("snapshot_date").isNull().cast("int")).alias("null_snapshot_date"),
    F.sum(F.col("date_key").isNull().cast("int")).alias("null_date_key")
).show()

# 3. Check date dimension key uniqueness
date_dim = spark.table(
    "workspace.youtube_trending.gold_dim_date"
)

print(
    "Duplicate date keys:",
    date_dim.groupBy("date_key").count()
        .filter(F.col("count") > 1).count()
)

# 4. Preview fact table
fact.orderBy(
    F.col("snapshot_date").desc()
).show(10, truncate=False)

gold_dim_date: 14 rows
gold_dim_region: 10 rows
gold_dim_category: 32 rows
gold_dim_channel: 2,214 rows
gold_dim_video: 3,698 rows
gold_fact_video_daily: 7,059 rows
+-------------+-----------+------------------+-------------+
|null_video_id|null_region|null_snapshot_date|null_date_key|
+-------------+-----------+------------------+-------------+
|            0|          0|                 0|            0|
+-------------+-----------+------------------+-------------+

Duplicate date keys: 0
+-----------+-----------+--------+-------------+-----------+----------------------------------------------------------------+--------+------+--------+--------+---------------------+-------------+----------------------------------------------------------------+
|video_id   |region_code|date_key|snapshot_date|category_id|channel_id_hash                                                 |views   |likes |dislikes|comments|engagement_rate      |batch_id     |row_hash                                          